In [2]:
# ═══════════════════════════════════════════════════════════════
#  CELL 1 — Imports & Global Configuration
# ═══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, joblib
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Paths ──────────────────────────────────────────────────────
NOTEBOOK_DIR        = '/Users/clttoolboxmacm1/Documents/1. Mechine Learning/llm_news'
DATA_PATH           = f'{NOTEBOOK_DIR}/digital_burnout_productivity_dataset.csv'
MODEL_BURNOUT_PATH  = f'{NOTEBOOK_DIR}/model_burnout.pkl'
MODEL_PROD_PATH     = f'{NOTEBOOK_DIR}/model_productivity.pkl'

# ── Feature Schema ─────────────────────────────────────────────
FEATURE_COLS = [
    'age', 'daily_screen_time', 'social_media_hours', 'doomscrolling_duration',
    'app_switch_frequency', 'notification_count', 'smartphone_unlocks',
    'late_night_device_usage', 'focus_sessions', 'deep_work_hours',
    'distraction_frequency', 'task_completion_rate', 'concentration_score',
    'sleep_hours', 'sleep_quality', 'caffeine_intake', 'physical_activity',
    'stress_level', 'workspace_quality', 'meeting_hours', 'internet_stability',
    'remote_work_days', 'motivation_level', 'mental_fatigue', 'emotional_exhaustion',
    'work_satisfaction', 'occupation', 'work_mode', 'device_usage_type',
]
CAT_COLS = ['occupation', 'work_mode', 'device_usage_type']
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

print("✅ Library & konfigurasi berhasil dimuat!")
print(f"   📌 Total fitur: {len(FEATURE_COLS)}  |  Numerik: {len(NUM_COLS)}  |  Kategorikal: {len(CAT_COLS)}")

✅ Library & konfigurasi berhasil dimuat!
   📌 Total fitur: 29  |  Numerik: 26  |  Kategorikal: 3


In [3]:
# ═══════════════════════════════════════════════════════════════
#  CELL 2 — Data Loading & ML Model Training
# ═══════════════════════════════════════════════════════════════

def classify_burnout(score):
    """Konversi skor burnout 0-100 menjadi label kelas."""
    if score >= 65:   return 'Tinggi'
    elif score >= 35: return 'Sedang'
    else:             return 'Rendah'

def make_preprocessor():
    """Buat ColumnTransformer: StandardScaler + OrdinalEncoder."""
    return ColumnTransformer([
        ('num', StandardScaler(), NUM_COLS),
        ('cat', OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1
        ), CAT_COLS),
    ], remainder='drop')

# ── Load or train models ───────────────────────────────────────
if os.path.exists(MODEL_BURNOUT_PATH) and os.path.exists(MODEL_PROD_PATH):
    print("📦 Memuat model yang sudah tersimpan…")
    model_burnout     = joblib.load(MODEL_BURNOUT_PATH)
    model_productivity = joblib.load(MODEL_PROD_PATH)
    print("✅ Model berhasil dimuat! (skip training)")

else:
    print("📂 Memuat dataset — sampel 100.000 baris dari 5 juta record…")
    df_raw   = pd.read_csv(DATA_PATH, nrows=600_000)
    df       = df_raw.sample(n=100_000, random_state=42).reset_index(drop=True)
    print(f"✅ Dataset dimuat: {df.shape}")

    # Target labels
    PROD_MAP = {'High': 'Tinggi', 'Medium': 'Sedang', 'Low': 'Rendah'}
    df['burnout_class']      = df['burnout_risk'].apply(classify_burnout)
    df['productivity_class'] = df['productivity_category'].map(PROD_MAP)

    # Feature matrix
    X          = df[FEATURE_COLS].copy()
    X[CAT_COLS] = X[CAT_COLS].astype(str)
    valid       = X.notna().all(axis=1)
    X           = X[valid]

    y_burnout     = df.loc[X.index, 'burnout_class']
    y_productivity = df.loc[X.index, 'productivity_class']

    # Split
    idx_all               = X.index.tolist()
    idx_tr, idx_te        = train_test_split(idx_all, test_size=0.2, random_state=42)
    X_tr, X_te            = X.loc[idx_tr], X.loc[idx_te]
    yb_tr, yb_te          = y_burnout.loc[idx_tr],     y_burnout.loc[idx_te]
    yp_tr, yp_te          = y_productivity.loc[idx_tr], y_productivity.loc[idx_te]

    # ── Train Burnout Model ──
    print("\n🔧 Melatih model Burnout Risk (RandomForest)…")
    model_burnout = Pipeline([
        ('prep', make_preprocessor()),
        ('clf',  RandomForestClassifier(
            n_estimators=80, max_depth=14, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
    ])
    model_burnout.fit(X_tr, yb_tr)
    acc_b = accuracy_score(yb_te, model_burnout.predict(X_te))
    print(f"   ✅ Akurasi Burnout Risk   : {acc_b:.2%}")

    # ── Train Productivity Model ──
    print("\n🔧 Melatih model Produktivitas (RandomForest)…")
    model_productivity = Pipeline([
        ('prep', make_preprocessor()),
        ('clf',  RandomForestClassifier(
            n_estimators=80, max_depth=14, min_samples_leaf=5,
            class_weight='balanced', random_state=42, n_jobs=-1
        )),
    ])
    model_productivity.fit(X_tr, yp_tr)
    acc_p = accuracy_score(yp_te, model_productivity.predict(X_te))
    print(f"   ✅ Akurasi Produktivitas  : {acc_p:.2%}")

    # Save
    joblib.dump(model_burnout,     MODEL_BURNOUT_PATH)
    joblib.dump(model_productivity, MODEL_PROD_PATH)
    print(f"\n💾 Model disimpan → {NOTEBOOK_DIR}")

print("\n🚀 ML Engine siap digunakan!")

📦 Memuat model yang sudah tersimpan…
✅ Model berhasil dimuat! (skip training)

🚀 ML Engine siap digunakan!


In [4]:
# ═══════════════════════════════════════════════════════════════
#  CELL 3 — Profile Matching Engine (GAP Analysis)
# ═══════════════════════════════════════════════════════════════

# ── Profil Ideal ───────────────────────────────────────────────
IDEAL_PROFILE = {
    'daily_screen_time':      6.0,
    'social_media_hours':     1.0,
    'doomscrolling_duration': 0.3,
    'notification_count':     60,
    'app_switch_frequency':   70,
    'smartphone_unlocks':     50,
    'late_night_device_usage': 0,
    'focus_sessions':         6,
    'deep_work_hours':        5.0,
    'distraction_frequency':  20,
    'task_completion_rate':   85,
    'concentration_score':    8,
    'sleep_hours':            8.0,
    'sleep_quality':          8,
    'caffeine_intake':        2,
    'physical_activity':      2.5,
    'stress_level':           3,
    'workspace_quality':      8,
    'meeting_hours':          2.0,
    'internet_stability':     8,
    'remote_work_days':       3,
    'motivation_level':       8.0,
    'mental_fatigue':         3,
    'emotional_exhaustion':   3,
    'work_satisfaction':      8,
}

# ── Rentang nilai tiap fitur (min, max) ────────────────────────
FEATURE_RANGES = {
    'daily_screen_time':      (1.0,  18.0),
    'social_media_hours':     (0.0,  12.0),
    'doomscrolling_duration': (0.0,   7.9),
    'notification_count':     (20,   399),
    'app_switch_frequency':   (10,   249),
    'smartphone_unlocks':     (15,   299),
    'late_night_device_usage':(0,      1),
    'focus_sessions':         (0,      9),
    'deep_work_hours':        (0.0,  11.8),
    'distraction_frequency':  (1,    119),
    'task_completion_rate':   (40,   100),
    'concentration_score':    (1,     10),
    'sleep_hours':            (3.0,  10.0),
    'sleep_quality':          (1,     10),
    'caffeine_intake':        (0,      7),
    'physical_activity':      (0.0,   5.0),
    'stress_level':           (1,     10),
    'workspace_quality':      (1,     10),
    'meeting_hours':          (0.0,  10.0),
    'internet_stability':     (1,     10),
    'remote_work_days':       (0,      6),
    'motivation_level':       (1.0,  10.0),
    'mental_fatigue':         (1,     10),
    'emotional_exhaustion':   (1,     10),
    'work_satisfaction':      (1,     10),
}

# ── Bobot fitur (total ≈ 1.0) ─────────────────────────────────
FEATURE_WEIGHTS = {
    'sleep_hours':            0.08,
    'sleep_quality':          0.07,
    'mental_fatigue':         0.07,
    'stress_level':           0.07,
    'deep_work_hours':        0.06,
    'focus_sessions':         0.06,
    'motivation_level':       0.06,
    'social_media_hours':     0.05,
    'emotional_exhaustion':   0.05,
    'task_completion_rate':   0.05,
    'concentration_score':    0.05,
    'distraction_frequency':  0.04,
    'physical_activity':      0.04,
    'doomscrolling_duration': 0.03,
    'work_satisfaction':      0.04,
    'daily_screen_time':      0.03,
    'notification_count':     0.03,
    'workspace_quality':      0.03,
    'caffeine_intake':        0.02,
    'app_switch_frequency':   0.02,
    'smartphone_unlocks':     0.02,
    'late_night_device_usage':0.02,
    'meeting_hours':          0.02,
    'internet_stability':     0.01,
    'remote_work_days':       0.01,
}

# ── Fitur: nilai lebih rendah = lebih baik ────────────────────
LOWER_IS_BETTER = {
    'daily_screen_time', 'social_media_hours', 'doomscrolling_duration',
    'notification_count', 'app_switch_frequency', 'smartphone_unlocks',
    'late_night_device_usage', 'distraction_frequency', 'caffeine_intake',
    'stress_level', 'mental_fatigue', 'emotional_exhaustion', 'meeting_hours',
}

# ── Label display ──────────────────────────────────────────────
FEAT_LABELS = {
    'daily_screen_time':      '🖥 Screen Time',
    'social_media_hours':     '📱 Media Sosial',
    'doomscrolling_duration': '📜 Doomscrolling',
    'notification_count':     '🔔 Notifikasi',
    'app_switch_frequency':   '🔀 App Switch',
    'smartphone_unlocks':     '🔓 Buka HP',
    'late_night_device_usage':'🌙 HP Malam',
    'focus_sessions':         '🎯 Sesi Fokus',
    'deep_work_hours':        '💡 Deep Work',
    'distraction_frequency':  '😵 Distraksi',
    'task_completion_rate':   '✅ Task Completion',
    'concentration_score':    '🔍 Konsentrasi',
    'sleep_hours':            '😴 Jam Tidur',
    'sleep_quality':          '⭐ Kualitas Tidur',
    'caffeine_intake':        '☕ Kafein',
    'physical_activity':      '🏃 Aktivitas Fisik',
    'stress_level':           '😰 Level Stres',
    'workspace_quality':      '🖥 Workspace',
    'meeting_hours':          '📅 Meeting',
    'internet_stability':     '📶 Internet',
    'remote_work_days':       '🏡 Remote Days',
    'motivation_level':       '🚀 Motivasi',
    'mental_fatigue':         '🧠 Kelelahan Mental',
    'emotional_exhaustion':   '💔 Kelelahan Emosional',
    'work_satisfaction':      '😊 Kepuasan Kerja',
}

# ── Core GAP Analysis ─────────────────────────────────────────
def _norm(feat: str, val: float) -> float:
    lo, hi = FEATURE_RANGES.get(feat, (0, 10))
    return (val - lo) / (hi - lo + 1e-9)

def compute_gap_analysis(user_data: dict):
    """
    Profile Matching GAP Analysis.
    Returns:
        gaps       – raw gap per feature (positive = user WORSE than ideal)
        gap_score  – aggregated deviation score 0–100 (higher = worse)
        top_issues – [(feat, gap, weighted_gap), …] sorted by severity
    """
    gaps, weighted_gaps = {}, {}
    total_weight = sum(FEATURE_WEIGHTS.values())

    for feat, ideal in IDEAL_PROFILE.items():
        if feat not in user_data:
            continue
        u_n   = _norm(feat, user_data[feat])
        id_n  = _norm(feat, ideal)
        # positive gap → user is worse than ideal
        gap   = (u_n - id_n) if feat in LOWER_IS_BETTER else (id_n - u_n)
        gaps[feat]          = gap
        weighted_gaps[feat] = max(0.0, gap) * FEATURE_WEIGHTS.get(feat, 0.02)

    gap_score  = sum(weighted_gaps.values()) / total_weight * 100
    top_issues = sorted(weighted_gaps.items(), key=lambda x: x[1], reverse=True)
    top_issues = [(f, gaps[f], wg) for f, wg in top_issues if wg > 0]
    return gaps, gap_score, top_issues

# ── Recommendation Generator ──────────────────────────────────
_REC = {
    'sleep_hours':
        lambda v, i: f"Tingkatkan jam tidur dari <b>{v:.1f}</b> ke <b>{i:.0f} jam/malam</b>. Tidur cukup meningkatkan konsentrasi & daya ingat hingga 40%.",
    'sleep_quality':
        lambda v, i: f"Kualitas tidur Anda <b>{v}/10</b>. Hindari layar & kafein 2 jam sebelum tidur. Target kualitas <b>{i:.0f}/10</b>.",
    'mental_fatigue':
        lambda v, i: f"Kelelahan mental tinggi (<b>{v}/10</b>). Jadwalkan micro-break 5 menit setiap 50 menit kerja — Teknik Pomodoro.",
    'stress_level':
        lambda v, i: f"Stres <b>{v}/10</b> melewati ambang aman. Praktikkan pernapasan diafragma atau meditasi <b>10 menit/hari</b>.",
    'deep_work_hours':
        lambda v, i: f"Deep work hanya <b>{v:.1f} jam/hari</b>. Blokir 2–4 jam pagi khusus kerja fokus. Target <b>{i:.0f} jam</b>.",
    'focus_sessions':
        lambda v, i: f"Tingkatkan sesi fokus dari <b>{v}</b> ke <b>{i:.0f}/hari</b>. Matikan semua notifikasi selama sesi berlangsung.",
    'motivation_level':
        lambda v, i: f"Motivasi <b>{v:.1f}/10</b>. Terapkan sistem reward setelah menyelesaikan tugas + SMART Goals mingguan.",
    'social_media_hours':
        lambda v, i: f"Kurangi media sosial dari <b>{v:.1f}</b> ke <b>{i:.0f} jam/hari</b>. Gunakan app timer atau mode grayscale.",
    'emotional_exhaustion':
        lambda v, i: f"Kelelahan emosional <b>{v}/10</b>. Luangkan waktu untuk hobi & social support. Coba journaling harian.",
    'task_completion_rate':
        lambda v, i: f"Completion rate <b>{v}%</b> perlu naik ke <b>{i:.0f}%</b>. Terapkan time-blocking dan prioritaskan 3 tugas utama/hari.",
    'concentration_score':
        lambda v, i: f"Konsentrasi <b>{v}/10</b>. Latih fokus: baca 30 menit/hari tanpa gangguan, tingkatkan secara bertahap.",
    'distraction_frequency':
        lambda v, i: f"Distraksi <b>{v}/hari</b> terlalu tinggi. Aktifkan 'Do Not Disturb' saat bekerja. Target di bawah <b>{i:.0f}/hari</b>.",
    'physical_activity':
        lambda v, i: f"Aktivitas fisik hanya <b>{v:.1f} jam/hari</b>. Target <b>{i:.1f} jam</b> — riset Harvard: meningkatkan produktivitas 23%.",
    'doomscrolling_duration':
        lambda v, i: f"Doomscrolling <b>{v:.1f} jam/hari</b> menguras energi mental. Batasi ke <b>{i:.1f} jam</b> — hapus app berita dari homescreen.",
    'work_satisfaction':
        lambda v, i: f"Kepuasan kerja <b>{v}/10</b>. Diskusikan target pengembangan dengan atasan; identifikasi aspek kerja yang menyenangkan.",
    'caffeine_intake':
        lambda v, i: f"Kafein <b>{v}/hari</b> berlebihan. Kurangi ke <b>{i:.0f}/hari</b> dan stop konsumsi setelah pukul 14:00.",
    'notification_count':
        lambda v, i: f"<b>{v} notifikasi/hari</b> sangat mengganggu. Matikan non-esensial — cek notifikasi hanya 3× sehari.",
    'workspace_quality':
        lambda v, i: f"Kualitas workspace <b>{v}/10</b>. Investasikan pada pencahayaan baik, kursi ergonomis, dan minimalisme visual.",
    'late_night_device_usage':
        lambda v, i: f"Penggunaan HP di malam hari merusak ritme sirkadian. Stop device <b>1 jam sebelum tidur</b>. Gunakan alarm fisik.",
    'meeting_hours':
        lambda v, i: f"Meeting <b>{v:.1f} jam/hari</b> menyita waktu produktif. Evaluasi mana yang bisa digantikan email. Target <b>{i:.0f} jam/hari</b>.",
}

def get_recommendations(top_issues: list, user_data: dict) -> list:
    recs = []
    for feat, gap, wgap in top_issues:
        if feat in _REC and feat in user_data and feat in IDEAL_PROFILE:
            recs.append(_REC[feat](user_data[feat], IDEAL_PROFILE[feat]))
        if len(recs) >= 6:
            break
    return recs

# ── Radar Chart ───────────────────────────────────────────────
_RADAR_KEYS = [
    ('sleep_hours',         'Tidur',       False),
    ('deep_work_hours',     'Deep Work',   False),
    ('motivation_level',    'Motivasi',    False),
    ('concentration_score', 'Konsentrasi', False),
    ('physical_activity',   'Aktivitas',   False),
    ('task_completion_rate','Completion',  False),
    ('work_satisfaction',   'Kepuasan',    False),
    ('stress_level',        'Stres↓',      True),
    ('social_media_hours',  'SosMed↓',     True),
    ('mental_fatigue',      'Kelelahan↓',  True),
]

def plot_radar(user_data: dict) -> plt.Figure:
    """Radar chart membandingkan profil pengguna vs profil ideal."""
    labels = [lbl for _, lbl, _ in _RADAR_KEYS]
    N      = len(labels)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    def to_pct(feat, val, inv):
        lo, hi = FEATURE_RANGES.get(feat, (0, 10))
        n = (val - lo) / (hi - lo + 1e-9)
        return (1 - n) if inv else n

    user_v  = [to_pct(f, user_data.get(f, IDEAL_PROFILE.get(f, 5)), inv)
               for f, _, inv in _RADAR_KEYS] + [None]
    ideal_v = [to_pct(f, IDEAL_PROFILE.get(f, 5), inv)
               for f, _, inv in _RADAR_KEYS] + [None]
    user_v[-1]  = user_v[0]
    ideal_v[-1] = ideal_v[0]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(projection='polar'))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#0d1b35')

    ax.plot(angles, ideal_v, '-o', lw=2,   color='#00d4aa', label='Profil Ideal', ms=5)
    ax.fill(angles, ideal_v,        alpha=0.15, color='#00d4aa')
    ax.plot(angles, user_v,  '-o', lw=2.5, color='#ff6b6b', label='Profil Anda',  ms=5)
    ax.fill(angles, user_v,         alpha=0.25, color='#ff6b6b')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, color='white', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['25%', '50%', '75%'], color='#777', fontsize=7)
    ax.grid(color='#444', alpha=0.4)
    ax.spines['polar'].set_color('#555')

    legend = ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.12),
                       facecolor='#1a1a2e', edgecolor='#555', labelcolor='white', fontsize=9)
    ax.set_title('Radar — Profil Anda vs Profil Ideal',
                 color='white', fontsize=12, fontweight='bold', pad=18)
    plt.tight_layout()
    return fig

print("✅ Mesin SPK Profile Matching siap!")
print(f"   📌 {len(IDEAL_PROFILE)} variabel referensi  |  {len(FEATURE_WEIGHTS)} bobot terkonfigurasi")

✅ Mesin SPK Profile Matching siap!
   📌 25 variabel referensi  |  25 bobot terkonfigurasi


In [5]:
# ═══════════════════════════════════════════════════════════════
#  CELL 4 — Widget Definitions & Form Layout
# ═══════════════════════════════════════════════════════════════

_sl  = {'description_width': '195px'}
_wl  = widgets.Layout(width='440px')
_wlm = widgets.Layout(width='440px')

# ── Digital Behavior ──────────────────────────────────────────
w_screen_time = widgets.FloatSlider(
    8.0, min=1.0, max=18.0, step=0.1,
    description='🖥 Screen Time (jam):', style=_sl, layout=_wl)
w_social = widgets.FloatSlider(
    3.5, min=0.0, max=12.0, step=0.1,
    description='📱 Media Sosial (jam):', style=_sl, layout=_wl)
w_doomscroll = widgets.FloatSlider(
    1.8, min=0.0, max=7.9, step=0.1,
    description='📜 Doomscrolling (jam):', style=_sl, layout=_wl)
w_notif = widgets.IntSlider(
    209, min=20, max=399, step=5,
    description='🔔 Notifikasi/hari:', style=_sl, layout=_wl)
w_unlocks = widgets.IntSlider(
    157, min=15, max=299, step=5,
    description='🔓 Buka HP/hari:', style=_sl, layout=_wl)
w_app_switch = widgets.IntSlider(
    129, min=10, max=249, step=5,
    description='🔀 App Switch/hari:', style=_sl, layout=_wl)
w_late_night = widgets.RadioButtons(
    options=[('Ya (1)', 1), ('Tidak (0)', 0)], value=1,
    description='🌙 HP di malam hari?', style=_sl)

# ── Sleep & Recovery ──────────────────────────────────────────
w_sleep_hrs = widgets.FloatSlider(
    6.5, min=3.0, max=10.0, step=0.1,
    description='😴 Jam Tidur/malam:', style=_sl, layout=_wl)
w_sleep_q = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='⭐ Kualitas Tidur (1-10):', style=_sl, layout=_wl)
w_fatigue = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='🧠 Kelelahan Mental (1-10):', style=_sl, layout=_wl)
w_emotion = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='💔 Kelelahan Emosional:', style=_sl, layout=_wl)
w_stress = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='😰 Level Stres (1-10):', style=_sl, layout=_wl)

# ── Work Performance ──────────────────────────────────────────
w_focus = widgets.IntSlider(
    4, min=0, max=9, step=1,
    description='🎯 Sesi Fokus/hari:', style=_sl, layout=_wl)
w_deep = widgets.FloatSlider(
    3.0, min=0.0, max=11.8, step=0.2,
    description='💡 Deep Work (jam):', style=_sl, layout=_wl)
w_distract = widgets.IntSlider(
    60, min=1, max=119, step=1,
    description='😵 Frekuensi Distraksi:', style=_sl, layout=_wl)
w_completion = widgets.IntSlider(
    70, min=40, max=100, step=1,
    description='✅ Task Completion (%):', style=_sl, layout=_wl)
w_concentration = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='🔍 Konsentrasi (1-10):', style=_sl, layout=_wl)
w_meeting = widgets.FloatSlider(
    2.5, min=0.0, max=10.0, step=0.5,
    description='📅 Meeting (jam/hari):', style=_sl, layout=_wl)

# ── Lifestyle ─────────────────────────────────────────────────
w_physical = widgets.FloatSlider(
    1.2, min=0.0, max=5.0, step=0.1,
    description='🏃 Aktivitas Fisik (jam):', style=_sl, layout=_wl)
w_caffeine = widgets.IntSlider(
    3, min=0, max=7, step=1,
    description='☕ Konsumsi Kafein/hari:', style=_sl, layout=_wl)
w_motivation = widgets.FloatSlider(
    5.5, min=1.0, max=10.0, step=0.1,
    description='🚀 Motivasi (1-10):', style=_sl, layout=_wl)
w_work_sat = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='😊 Kepuasan Kerja (1-10):', style=_sl, layout=_wl)

# ── Profile & Context ─────────────────────────────────────────
w_age = widgets.IntSlider(
    30, min=18, max=59, step=1,
    description='🎂 Usia:', style=_sl, layout=_wl)
w_occupation = widgets.Dropdown(
    options=['Software Engineer', 'Analyst', 'Manager', 'Designer',
             'Freelancer', 'Student', 'Content Creator'],
    value='Software Engineer',
    description='👔 Pekerjaan:', style=_sl, layout=_wl)
w_work_mode = widgets.Dropdown(
    options=['Office', 'Hybrid', 'Remote'],
    value='Hybrid',
    description='🏠 Mode Kerja:', style=_sl, layout=_wl)
w_device_type = widgets.Dropdown(
    options=['Work-Centric', 'Balanced', 'Entertainment-Centric'],
    value='Balanced',
    description='📲 Tipe Penggunaan Device:', style=_sl, layout=_wl)
w_workspace_q = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='🖼 Kualitas Workspace (1-10):', style=_sl, layout=_wl)
w_remote_days = widgets.IntSlider(
    3, min=0, max=6, step=1,
    description='🏡 Hari Remote/minggu:', style=_sl, layout=_wl)
w_internet = widgets.IntSlider(
    5, min=1, max=10, step=1,
    description='📶 Stabilitas Internet (1-10):', style=_sl, layout=_wl)

# ── Action Button & Output ─────────────────────────────────────
btn_analyze = widgets.Button(
    description='🔍  Analisis Produktivitas Saya',
    button_style='danger',
    layout=widgets.Layout(width='340px', height='50px'),
)
output_panel = widgets.Output()

print("✅ Semua widget berhasil didefinisikan!")
print("   ▶ Jalankan sel berikutnya untuk menampilkan dashboard.")

✅ Semua widget berhasil didefinisikan!
   ▶ Jalankan sel berikutnya untuk menampilkan dashboard.


In [6]:
# ═══════════════════════════════════════════════════════════════
#  CELL 5 — Callback Engine + Dashboard Launch
# ═══════════════════════════════════════════════════════════════

def _section_hdr(title: str) -> widgets.HTML:
    return widgets.HTML(
        f'<div style="background:#0f3460;color:white;padding:7px 15px;'
        f'border-radius:7px;margin:10px 0 3px;font-weight:bold;font-size:13px;'
        f'border-left:4px solid #e94560;">{title}</div>'
    )

def _badge(label: str, value: str, color: str) -> str:
    return (
        f'<div style="flex:1;min-width:150px;background:rgba(255,255,255,.06);'
        f'border-radius:12px;padding:16px 18px;border-top:3px solid {color};text-align:center;">'
        f'<div style="font-size:10px;color:#aaa;letter-spacing:1px;text-transform:uppercase;'
        f'margin-bottom:6px">{label}</div>'
        f'<div style="font-size:26px;font-weight:900;color:{color}">{value}</div>'
        f'</div>'
    )

def collect_user_data() -> dict:
    return {
        'age':                     w_age.value,
        'daily_screen_time':       w_screen_time.value,
        'social_media_hours':      w_social.value,
        'doomscrolling_duration':  w_doomscroll.value,
        'app_switch_frequency':    w_app_switch.value,
        'notification_count':      w_notif.value,
        'smartphone_unlocks':      w_unlocks.value,
        'late_night_device_usage': w_late_night.value,
        'focus_sessions':          w_focus.value,
        'deep_work_hours':         w_deep.value,
        'distraction_frequency':   w_distract.value,
        'task_completion_rate':    w_completion.value,
        'concentration_score':     w_concentration.value,
        'sleep_hours':             w_sleep_hrs.value,
        'sleep_quality':           w_sleep_q.value,
        'caffeine_intake':         w_caffeine.value,
        'physical_activity':       w_physical.value,
        'stress_level':            w_stress.value,
        'workspace_quality':       w_workspace_q.value,
        'meeting_hours':           w_meeting.value,
        'internet_stability':      w_internet.value,
        'remote_work_days':        w_remote_days.value,
        'motivation_level':        w_motivation.value,
        'mental_fatigue':          w_fatigue.value,
        'emotional_exhaustion':    w_emotion.value,
        'work_satisfaction':       w_work_sat.value,
        'occupation':              w_occupation.value,
        'work_mode':               w_work_mode.value,
        'device_usage_type':       w_device_type.value,
    }

def on_analyze_clicked(_):
    with output_panel:
        clear_output(wait=True)
        display(HTML(
            '<div style="color:#aaa;padding:12px 0">⏳ Menganalisis data Anda…</div>'
        ))

        # ── 1. Kumpulkan input ──────────────────────────────────
        ud = collect_user_data()
        input_df = pd.DataFrame([ud])[FEATURE_COLS]
        input_df[CAT_COLS] = input_df[CAT_COLS].astype(str)

        # ── 2. Prediksi ML ─────────────────────────────────────
        burn_pred  = model_burnout.predict(input_df)[0]
        prod_pred  = model_productivity.predict(input_df)[0]
        burn_proba = dict(zip(model_burnout.classes_,
                              model_burnout.predict_proba(input_df)[0]))
        prod_proba = dict(zip(model_productivity.classes_,
                              model_productivity.predict_proba(input_df)[0]))

        # ── 3. GAP Analysis (SPK) ───────────────────────────────
        gaps, gap_score, top_issues = compute_gap_analysis(ud)
        recs = get_recommendations(top_issues, ud)

        # ── 4. Warna kontekstual ────────────────────────────────
        RC = {'Tinggi': '#ff4d4d', 'Sedang': '#ffaa00', 'Rendah': '#44c98a'}
        PC = {'Tinggi': '#44c98a', 'Sedang': '#ffaa00', 'Rendah': '#ff4d4d'}
        bc = RC.get(burn_pred, '#888')
        pc = PC.get(prod_pred, '#888')
        gc = '#ff4d4d' if gap_score > 60 else ('#ffaa00' if gap_score > 30 else '#44c98a')

        # ── 5. Alert banner ─────────────────────────────────────
        _alerts = {
            'Tinggi': (
                '⚠️ PERINGATAN KRITIS',
                f'Risiko burnout Anda sangat <b>TINGGI</b>. Segera evaluasi beban kerja dan '
                f'pola hidup Anda. Pertimbangkan konsultasi profesional jika kondisi berlanjut.'),
            'Sedang': (
                '⚡ PERHATIAN',
                f'Risiko burnout <b>SEDANG</b>. Mulai terapkan perubahan gaya hidup '
                f'sebelum kondisi memburuk.'),
            'Rendah': (
                '✅ STATUS BAIK',
                f'Risiko burnout <b>RENDAH</b>. Pertahankan kebiasaan positif yang sudah ada!'),
        }
        al_title, al_msg = _alerts.get(burn_pred, _alerts['Sedang'])

        # ── 6. Bangun HTML Output ──────────────────────────────
        html = f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;
            background:linear-gradient(135deg,#1a1a2e,#16213e);
            border-radius:16px;padding:28px;color:white;
            border:1px solid #0f3460;max-width:960px;margin:8px 0">

  <h2 style="text-align:center;color:#e94560;margin:0 0 22px;font-size:20px">
    📊 Hasil Analisis Sistem Pendukung Keputusan Produktivitas
  </h2>

  <!-- ─ Badges ─────────────────────────────────────────── -->
  <div style="display:flex;gap:12px;justify-content:center;flex-wrap:wrap;margin-bottom:20px">
    {_badge('Risiko Burnout',  burn_pred.upper(),                  bc)}
    {_badge('Probabilitas ML', f"{burn_proba.get(burn_pred,0):.0%}", bc)}
    {_badge('Produktivitas',   prod_pred.upper(),                  pc)}
    {_badge('GAP Score SPK',   f"{gap_score:.1f} / 100",           gc)}
  </div>

  <!-- ─ Alert ──────────────────────────────────────────── -->
  <div style="background:rgba(255,255,255,.05);border-radius:10px;
              padding:13px 18px;margin-bottom:20px;border-left:4px solid {bc}">
    <span style="color:{bc};font-weight:bold">{al_title}:</span> {al_msg}
  </div>
"""
        # ── Top Issues ──
        if top_issues:
            html += """
  <h3 style="color:#e94560;border-bottom:1px solid #333;padding-bottom:6px;margin-bottom:12px">
    🔍 Faktor Penyumbang GAP Terbesar — Profile Matching SPK
  </h3>
  <div style="display:flex;flex-wrap:wrap;gap:10px;margin-bottom:22px">
"""
            for feat, gap, wgap in top_issues[:6]:
                lbl   = FEAT_LABELS.get(feat, feat)
                clr   = '#ff4d4d' if wgap > 0.025 else '#ffaa00'
                html += (
                    f'<div style="background:rgba(255,255,255,.06);border-radius:8px;'
                    f'padding:10px 14px;border-top:2px solid {clr};min-width:140px">'
                    f'<div style="font-size:13px;font-weight:bold">{lbl}</div>'
                    f'<div style="font-size:11px;color:{clr};margin-top:3px">'
                    f'Deviasi: {abs(gap)*100:.0f}% dari ideal</div>'
                    f'</div>'
                )
            html += '  </div>\n'

        # ── Recommendations ──
        if recs:
            html += """
  <h3 style="color:#00d4aa;border-bottom:1px solid #333;padding-bottom:6px;margin-bottom:12px">
    💡 Rekomendasi Personalisasi (Berbasis Analisis GAP)
  </h3>
  <div>
"""
            for i, rec in enumerate(recs, 1):
                html += (
                    f'<div style="background:rgba(0,212,170,.07);border-left:3px solid #00d4aa;'
                    f'padding:11px 16px;margin-bottom:8px;border-radius:0 8px 8px 0;">'
                    f'<span style="color:#00d4aa;font-weight:bold">#{i}</span> {rec}</div>'
                )
            html += '  </div>\n'

        html += '</div>'

        # ── 7. Tampilkan HTML + Radar Chart ────────────────────
        clear_output(wait=True)
        display(HTML(html))

        fig = plot_radar(ud)
        plt.show()
        plt.close(fig)

# ── Registrasi callback ────────────────────────────────────────
btn_analyze.on_click(on_analyze_clicked)

# ── Layout UI ─────────────────────────────────────────────────
HEADER = widgets.HTML("""
<div style="background:linear-gradient(135deg,#1a1a2e,#0f3460);
            border-radius:14px;padding:24px;text-align:center;
            border:1px solid #e94560;margin-bottom:14px">
  <h1 style="color:#e94560;margin:0 0 6px;font-size:21px">
    🧠 Sistem Pendukung Keputusan Cerdas — Personalisasi Produktivitas
  </h1>
  <p style="color:#9aa;margin:0;font-size:12px">
    Machine Learning (RandomForest) + Profile Matching SPK &nbsp;|&nbsp;
    Dataset: 5 Juta Record · 34 Kolom · Digital Burnout & Productivity Analytics
  </p>
</div>
""")

col_left = widgets.VBox([
    _section_hdr('📱 Digital Behavior'),
    w_screen_time, w_social, w_doomscroll,
    w_notif, w_unlocks, w_app_switch, w_late_night,
    _section_hdr('😴 Sleep & Recovery'),
    w_sleep_hrs, w_sleep_q, w_fatigue, w_emotion, w_stress,
], layout=widgets.Layout(width='480px', padding='6px'))

col_right = widgets.VBox([
    _section_hdr('💼 Work Performance'),
    w_focus, w_deep, w_distract, w_completion, w_concentration, w_meeting,
    _section_hdr('🏃 Lifestyle'),
    w_physical, w_caffeine, w_motivation, w_work_sat,
    _section_hdr('👔 Profil & Konteks Kerja'),
    w_age, w_occupation, w_work_mode, w_device_type,
    w_workspace_q, w_remote_days, w_internet,
], layout=widgets.Layout(width='480px', padding='6px'))

form_row = widgets.HBox(
    [col_left, col_right],
    layout=widgets.Layout(justify_content='center')
)
btn_row = widgets.HBox(
    [btn_analyze],
    layout=widgets.Layout(justify_content='center', margin='16px 0 8px')
)

FULL_UI = widgets.VBox([HEADER, form_row, btn_row, output_panel])
display(FULL_UI)